# Лабораторная №6
## Вариант 10

## Задание 1
 Загрузите исходные данные (csv-файл архива технологических сообщений). Запустите алгоритм поиска ассоциативных правил.

In [48]:
from mlxtend.frequent_patterns import apriori, association_rules 
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd


In [49]:
df = pd.read_csv('asutp_alarm_log_ansi.csv', encoding='ansi')

In [50]:
df.head()

,Timestamp,Event_Code,Description
0,2026-01-01 00:12:00,E01,Высокий уровень в сепараторе V-101 (LSHH)
1,2026-01-01 00:38:00,E102,Высокая температура подшипника (TSHH)
2,2026-01-01 00:38:00,E114,Срабатывание кнопки ESD (Грибок)
3,2026-01-01 00:40:20,E115,Полное обесточивание установки
4,2026-01-01 00:40:28,E103,Высокая вибрация ротора


In [51]:
te = TransactionEncoder()
te_ary = te.fit(df.values).transform(df.values)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

In [52]:
print(f"Размерность матрицы транзакций: {df_encoded.shape}")
print(df_encoded.head())

Размерность матрицы транзакций: (9657, 9629)
   2026-01-01 00:12:00  2026-01-01 00:38:00  2026-01-01 00:40:20  \
0                 True                False                False   
1                False                 True                False   
2                False                 True                False   
3                False                False                 True   
4                False                False                False   

   2026-01-01 00:40:28  2026-01-01 01:15:00  2026-01-01 01:15:34  \
0                False                False                False   
1                False                False                False   
2                False                False                False   
3                False                False                False   
4                 True                False                False   

   2026-01-01 01:18:00  2026-01-01 01:18:14  2026-01-01 01:29:00  \
0                False                False                False   
1

In [54]:
frequent_itemsets = apriori(df_encoded, min_support=0.001, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

print(f"Найдено правил: {len(rules)}")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].iloc[9])

Найдено правил: 132
antecedents                        frozenset({E05})
consequents    frozenset({Высокая вибрация ротора})
support                                    0.021228
confidence                                      1.0
lift                                      30.951923
Name: 9, dtype: object


## Задание 2. 
Выполните по вариантам (номер строки таблицы-результата выполнения задания 1= вариант) интерпретацию ассоциативных правил (по их характеристикам: поддержка, достоверность, лифт). Определите тип правила (например: тривиальное технологическое, скрытая корреляция, ложное срабатывание). Дайте рекомендации оператору или инженеру по использованию правила.

### E05 -> Высокая вибрация ротора
antecedents -                       E05\
consequents -  Высокая вибрация ротора\
support -                      0.021228 - 2% - достаточно частое\
confidence -                        1.0 - стопроцентная достоверность - E05 всегда предшествовало высокой вибрации в некотором временном окне\
lift -                        30.951923 - вибрация ротора появляется в 31 раз чаще с E05, чем без этого события - очень сильная зависимость

Тип: Скрытая корреляция с высокой предсказательной ценностью\
Скрытая - связь не очевидна (что такое E05)\
Корреляция - lift = 31 и confidence = 100%\
Предсказательная ценность - 2% от всех событий, которые можно четко классифицировать как предвестника вибрации ротора\
Оператору: оценивать Е05 как предвестника аварии, предпринять меры по избежанию высокой вибрации ротора\
Инженеру: внедрить предиктивную логику, убрать другие уведомления о проблеме с высокой вибрации ротора

## Задание 3. 
Запустите алгоритм apriori на другом диапазоне допустимой достоверности (от 25% до 40%). Проведите интерпретацию новых полученных правил. Какой вывод можно сделать в том случае, если в логах нет событий в указанном диапазоне достоверности?

In [55]:
rules_low_conf = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.25)
rules_low_conf = rules_low_conf[rules_low_conf["confidence"] <= 0.40]
print(f"Правил с confidence 0.25-0.40: {len(rules_low_conf)}")
if len(rules_low_conf) > 0:
    display(rules_low_conf[['antecedents','consequents','support','confidence','lift']].head())

Правил с confidence 0.25-0.40: 1


,antecedents,consequents,support,confidence,lift
20,frozenset({Высокая вибрация ротора}),frozenset({E103}),0.01108,0.342949,30.951923


Существует только 1 такое правило, оно случайное, может значить и то, что у них есть общее начальное условия, \
или что они связаны косвенно через 3й параметр\
в остальном таких правил нет, и это нормально - в такой системе будут правило либо с высокой достоверностью - \
физически обоснованные правила, либо параметры с низкой confidence - случайный шум.

## Задание 4.1. 
Проведите анализ ассоциативных правил по варианту из таблиц 1 и 2.\
таблица 1\
Срабатывание кнопки ESD (Грибок) -> Полное обесточивание установки\
таблица 2\
Переход в режим "Регенерация" + Задвижка сырого газа не закрыта -> Блокировка запуска печи нагрева

In [64]:
ant_t1 = "Срабатывание кнопки ESD (Грибок)"
ant_t2 = ['Переход в режим "Регенерация"', "Задвижка сырого газа не закрыта"]
cons_t1 = "Полное обесточивание установки"
cons_t2 = "Блокировка запуска печи нагрева"
mask_ant_t1 = rules['antecedents'].apply(lambda x: ant_t1 in x)
mask_con_t1 = rules['consequents'].apply(lambda x: cons_t1 in x)
rules_t1 = rules[mask_ant_t1 & mask_con_t1].drop_duplicates().reset_index(drop=True)
print(f"Правил с участием '{ant_t1}' и '{cons_t1}': {len(rules_t1)}")
print(rules_t1[['antecedents','consequents','support','confidence','lift']])

mask_ant_t2 = rules['antecedents'].apply(lambda x: all(c in x for c in ant_t2))
mask_con_t2 = rules['consequents'].apply(lambda x: cons_t2 in x)
rules_t2 = rules[mask_ant_t2 & mask_con_t2].drop_duplicates().reset_index(drop=True)
print(f"Правил с участием '{ant_t2}' и '{cons_t2}': {len(rules_t2)}")
print(rules_t2[['antecedents','consequents','support','confidence','lift']])

Правил с участием 'Срабатывание кнопки ESD (Грибок)' и 'Полное обесточивание установки': 0
Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []
Правил с участием '['Переход в режим "Регенерация"', 'Задвижка сырого газа не закрыта']' и 'Блокировка запуска печи нагрева': 0
Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []


Нет таких правил.

## Задание 4.2. 
Ответьте на вопрос (по схеме «что-если»): что произойдет на установке, если сработает: 
а) 1 условие из транзакции А (таблица 1), \
б) 2 условия из транзакции Б (таблица 2).\
 Для ответа ориентируйтесь на все найденные в датасете правила, где ваше событие выступает Условием (antecedent).\
  На какие характеристики правила нужно опираться (Confidence или Support), чтобы предсказать развитие аварии?

In [73]:
rules_a = rules[mask_ant_t1]
rules_b = rules[mask_ant_t2]

print(f"Правила с условием «{ant_t1}»: {len(rules_a)}")
if not rules_a.empty: display(rules_a[['antecedents','consequents','confidence','lift', 'support']].head(3))

print(f"\nПравила с условием «{ant_t2[0]} + {ant_t2[1]}»: {len(rules_b)}")
if not rules_b.empty: display(rules_b[['antecedents','consequents','confidence','lift', 'support']].head(3))

Правила с условием «Срабатывание кнопки ESD (Грибок)»: 1


,antecedents,consequents,confidence,lift,support
42,frozenset({Срабатывание кнопки ESD (Грибок)}),frozenset({E114}),1.0,80.475,0.012426



Правила с условием «Переход в режим "Регенерация" + Задвижка сырого газа не закрыта»: 0


При срабатывании кнопки ESD происходит E114, это 100% точно происходит, в 80 раз E114 происходит с ESD чем без этого, важная корреляция.\
Со вторыми условиями нет никаких правил.

## Задание 5. 
Найдите правила с наибольшей:\
а) поддержкой,\
б) достоверностью, \
в) лифтом,\
г) левереджем (leverage).\
 Проведите их интерпретацию с точки зрения безопасности завода.

In [72]:
metrics = {
    "а) Поддержка (Support)": rules.loc[rules["support"].idxmax()],
    "б) Достоверность (Confidence)": rules.loc[rules["confidence"].idxmax()],
    "в) Лифт (Lift)": rules.loc[rules["lift"].idxmax()],
    "г) Левередж (Leverage)": rules.loc[rules["leverage"].idxmax()]
}

for name, row in metrics.items():
    print(f"\nМаксимальная {name}:")
    print(f"   Условие   -> {list(row['antecedents'])}")
    print(f"   Следствие -> {list(row['consequents'])}")
    print(f"   Support: {row['support']:.4f} | Conf: {row['confidence']:.2f} | Lift: {row['lift']:.1f} | Lev: {row['leverage']:.4f}")
    print()


Максимальная а) Поддержка (Support):
   Условие   -> ['N06']
   Следствие -> ['Срабатывание концевого выключателя задвижки']
   Support: 0.0388 | Conf: 1.00 | Lift: 25.8 | Lev: 0.0373


Максимальная б) Достоверность (Confidence):
   Условие   -> ['E01']
   Следствие -> ['Высокий уровень в сепараторе V-101 (LSHH)']
   Support: 0.0229 | Conf: 1.00 | Lift: 43.7 | Lev: 0.0224


Максимальная в) Лифт (Lift):
   Условие   -> ['Разрушение муфты (оповещение)']
   Следствие -> ['E137']
   Support: 0.0095 | Conf: 1.00 | Lift: 105.0 | Lev: 0.0094


Максимальная г) Левередж (Leverage):
   Условие   -> ['N06']
   Следствие -> ['Срабатывание концевого выключателя задвижки']
   Support: 0.0388 | Conf: 1.00 | Lift: 25.8 | Lev: 0.0373



#### Правило с максимальной поддержкой(оно же правило с наибольшим левередж):
- встречается чаще всего
- 100% достоверность
- правило должно учитываться, особенно учитывая что оно часто встречается, нужно его обрабатывать
#### Правило с максимальной достоверностью:
- 100% достоверность и 44 лифт - очень явная корреляция 
- правило должно учитываться, нужно его обрабатывать
#### Правило с максимальным лифтом:
- 100% достоверности и 105 делает это правило самым явным нужно его обрабатывать и учитывать


## Контрольные вопросы

1. В чем заключается основная задача анализа корреляции событий (алармов) на промышленном предприятии?\
**Основная задача** — снижение информационного шума (alarm flooding) и выявление скрытых причинно-следственных связей между алармами для:
- автоматической диагностики **корневой причины** отказа;
- настройки предиктивной логики ПАЗ;
- оптимизации реакции оператора.\
**К примеру**  
Правило `E05 -> Высокая вибрация ротора` (Confidence=100%, Lift=31) показывает, что код `E05` — надёжный предиктор вибрации. Без анализа ассоциаций оператор видел бы два разрозненных аларма; с анализом — понимает, что `E05` требует немедленной проверки подшипника для предотвращения аварии.

2. Что выступает в роли "транзакции" при анализе временных рядов и логов событий?
**В контексте работы:**  
Каждая **уникальная временная метка** в файле `asutp_alarm_log_ansi.csv` — это отдельная транзакция. Все события, зафиксированные в одну секунду, считаются "купленными вместе".
``` python
# Код из работы:
te_ary = te.fit(df.values).transform(df.values)
# Результат: матрица [9657 транзакций × 9629 уникальных событий]
```
3. Что показывают параметры Поддержка (Support) и Достоверность (Confidence)? На что опираться инженеру, если правило имеет поддержку 1%, но достоверность 99%?\

|Метрика|Формула|Интерпретация в данной работе|
|---|---|---|
|Support|P(A∩B) = доля транзакций с условием И следствием|Правило E05 -> Вибрация имеет Support=2.12% ->встречалось ~205 раз из 9657 окон. Для аварийных сценариев это нормальная частота.|
|Confidence|P(BIA) = доля транзакций с условием, где также есть следствие|Confidence=100% -> ни разу не было случая, чтобы E05 сработало без последующей вибрации.|
4. Лифт (Lift) и Левередж (Leverage). Как с их помощью определить, является ли срабатывание двух сигнализаций простым совпадением (независимыми событиями) или жесткой физической закономерностью?

|Метрика|Формула|Интерпретация|Порог для "закономерности"|
|-|-|-|-|
|Lift|P(BIA) / P(B)|Lift=1 -> независимые события. Lift>1 → положительная связь.|Lift > 3 -> сильная зависимость|
|Leverage|P(A∩B) - P(A)·P(B)|Leverage≈0 -> независимы. Leverage>0 -> чаще, чем случайно.|Leverage > 0.01|

5. Как результаты анализа ассоциативных правил могут помочь инженеру АСУ ТП разгрузить интерфейс (SCADA-систему) диспетчера завода?

Три практических механизма:

1. Агрегация алармов (Alarm Aggregation)

Если правило A → B имеет Confidence > 0.95:\
Показывать оператору только первичное событие A.\
Вторичное B скрывать или помечать как "следствие".\
Пример: Правило Срабатывание кнопки ESD → E114 (Conf=100%). При нажатии "грибка" диспетчер видит только ESD-актив, а код E114 сворачивается в комментарий.

2. Подавление шумовых алармов (Noise Suppression)

Правила с Lift ≈ 1 и низким Confidence — статистический шум. Их можно:\
Перевести в журнал без звукового оповещения.\
Показать только при ручном запросе.

3. Динамический приоритет (Dynamic Prioritization)

На основе Confidence и Lift настраивать приоритет:
IF (сработало правило с Confidence > 0.9) THEN:
   - Повысить приоритет условия до "Critical"
   - Понизить приоритет следствия до "Info"